In [ ]:
# Cell 1: Imports
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

from bs4 import BeautifulSoup
import time
import json
from datetime import datetime, timedelta
import re
import pandas as pd
import numpy as np
import random
from urllib.parse import urljoin

import undetected_chromedriver as uc

# For PostgreSQL
import psycopg2
from psycopg2 import sql
from psycopg2.extras import execute_values
from dotenv import load_dotenv
import os

In [ ]:
# Cell 2: Constants and Configuration

# --- URL và Scraping Params ---
TARGET_URL_BASE = "https://www.topcv.vn"
INITIAL_TARGET_URL = "https://www.topcv.vn/tim-viec-lam-moi-nhat-tai-ho-chi-minh-l2?type_keyword=0&sba=1&locations=l2"

MAX_PAGES_TO_SCRAPE = 1 # Chỉ cào 1 trang danh sách để test
# Đặt một số lớn để cố gắng cào hết các job trên trang đầu tiên, 
# hoặc bạn có thể bỏ qua biến này nếu vòng lặp for của bạn tự động duyệt hết các item tìm được
MAX_DETAILS_PER_PAGE = 3 # Đảm bảo nó lớn hơn số job thường có trên 1 trang (thường là 50)

# --- File Output ---
TIMESTAMP_STR = datetime.now().strftime('%Y%m%d_%H%M%S')
JSON_OUTPUT_FILENAME = f"topcv_detailed_jobs_data_{TIMESTAMP_STR}.json"

# --- HCM Districts ---
HCM_DISTRICTS_FULL_LIST = [
    "Quận 1", "Quận 2", "Quận 3", "Quận 4", "Quận 5", "Quận 6", "Quận 7", "Quận 8",
    "Quận 9", "Quận 10", "Quận 11", "Quận 12",
    "Thủ Đức", "Bình Thạnh", "Tân Bình", "Tân Phú", "Phú Nhuận", "Gò Vấp", "Bình Tân",
    "Hóc Môn", "Củ Chi", "Nhà Bè", "Bình Chánh", "Cần Giờ", "Thành phố Thủ Đức"
]
HCM_DISTRICTS_NORMALIZED_FOR_MATCH = [
    d.lower().replace("quận ", "").replace("huyện ", "").replace("thành phố ", "").replace("tp. ", "")
    for d in HCM_DISTRICTS_FULL_LIST
]

# --- Database Configuration ---
DB_TABLE_NAME = 'topcv_jobs_detailed'

print(f"Cell 2: Các hằng số và cấu hình đã được thiết lập.")
print(f"  URL ban đầu: {INITIAL_TARGET_URL}")
print(f"  Số trang tối đa cào: {MAX_PAGES_TO_SCRAPE}")
if 'MAX_DETAILS_PER_PAGE' in locals():
     print(f"  Số job chi tiết tối đa mỗi trang (nếu áp dụng): {MAX_DETAILS_PER_PAGE}")
print(f"  File JSON output: {JSON_OUTPUT_FILENAME}")
print(f"  Tên bảng DB: {DB_TABLE_NAME}")

In [ ]:
# Cell 10: Load Data from PostgreSQL into Pandas DataFrame

import pandas as pd
import os
import psycopg2
from dotenv import load_dotenv

load_dotenv()
DB_TABLE_NAME = 'topcv_jobs_detailed'

print(f"\nCell 10: Đang kết nối DB và tải dữ liệu từ bảng '{DB_TABLE_NAME}'...")

conn = None
df_jobs = pd.DataFrame() # Khởi tạo df rỗng

try:
    conn = psycopg2.connect(
        dbname=os.getenv('DB_NAME'),
        user=os.getenv('DB_USER'),
        password=os.getenv('DB_PASSWORD'),
        host=os.getenv('DB_HOST', 'localhost'),
        port=os.getenv('DB_PORT', '5432')
    )
    
    # Chỉ lấy những job đã crawl chi tiết thành công
    query = f"SELECT * FROM {DB_TABLE_NAME} WHERE status = 'completed';"
    df_jobs = pd.read_sql(query, conn)

    if not df_jobs.empty:
        print(f"  Đã tải thành công {len(df_jobs)} job items đã hoàn thành.")
        print("\n  Thông tin DataFrame (df_jobs.info()):")
        df_jobs.info() 
        print('\n  === 5 dòng dữ liệu đầu tiên (df_jobs.head()) ===')
        display(df_jobs.head())
    else:
        print("  Không có dữ liệu nào (status='completed') trong DB để phân tích.")

except Exception as e:
    print(f"  Lỗi khi tải dữ liệu từ DB: {e}")
finally:
    if conn:
        conn.close()

In [ ]:
# Cell 11: Data Processing Helper Functions (Salary, Experience from List Page)

def parse_salary_string_for_list_data(salary_text):
    """Parse salary string (thường từ trang list) thành min, max, currency, type."""
    if pd.isna(salary_text): return np.nan, np.nan, 'triệu VNĐ', 'Không xác định'
    text_original, text_lower = str(salary_text), str(salary_text).lower()
    min_sal, max_sal, currency, sal_type = np.nan, np.nan, 'triệu VNĐ', 'Không xác định'

    if "thoả thuận" in text_lower:
        sal_type = "Thoả thuận"; return min_sal, max_sal, currency, sal_type

    if "usd" in text_lower: currency = "USD"; text_extract = re.sub(r'[^0-9.,\-]', '', text_lower.replace("usd",""))
    elif "triệu" in text_lower: text_extract = re.sub(r'[^0-9.,\-]', '', text_lower.replace("triệu","").replace("vnd",""))
    else: text_extract = re.sub(r'[^0-9.,\-]', '', text_lower)
    
    nums_str = re.findall(r'\d+[\.,]?\d*', text_extract)
    nums_float = []
    for ns in nums_str:
        cleaned_ns = ns.replace('.', '').replace(',', '.') if ',' in ns and '.' not in ns[:ns.find(',')] else ns.replace(',', '')
        try: nums_float.append(float(cleaned_ns))
        except ValueError: pass
    nums = sorted(nums_float)

    if not nums: return min_sal, max_sal, currency, sal_type

    if "tới" in text_lower or "upto" in text_lower or ("đến" in text_lower and len(nums) == 1 and "từ" not in text_lower):
        sal_type = "Tối đa"; max_sal = nums[-1] if nums else np.nan
    elif "từ" in text_lower and not any(k in text_lower for k in ["đến", "tới"]) and "-" not in text_original:
        sal_type = "Tối thiểu"; min_sal = nums[0] if nums else np.nan
    elif (("-" in text_original or "đến" in text_lower) and len(nums) >= 2) or len(nums) >= 2: # Sửa điều kiện này để bắt cả trường hợp chỉ có 2 số mà không có "đến" hay "-"
        sal_type = "Khoảng"
        if len(nums) >= 2:
            min_sal, max_sal = nums[0], nums[-1]
        elif len(nums) == 1: # Nếu sau khi lọc chỉ còn 1 số cho "khoảng"
             min_sal = max_sal = nums[0]
             sal_type = "Cố định" # Hoặc "Khoảng không rõ ràng"
        if min_sal == max_sal and pd.notna(min_sal) : sal_type = "Cố định" # Nếu min = max thì là cố định
    elif len(nums) == 1:
        sal_type = "Cố định"; min_sal = max_sal = nums[0]
    return min_sal, max_sal, currency, sal_type

def standardize_experience_for_list_data(exp_str):
    """Chuẩn hóa experience string (thường từ trang list)."""
    if pd.isna(exp_str): return 'Không xác định'
    exp_lower = str(exp_str).lower()
    if any(k in exp_lower for k in ["không yêu cầu", "chưa có kinh nghiệm", "no experience", "fresher", "mới tốt nghiệp"]): return "Không yêu cầu"
    if m := re.search(r'(?:dưới|<)\s*(\d+)\s*(?:năm|year)', exp_lower): return f"< {m.group(1)} năm"
    if m := re.search(r'(?:từ\s*)?(\d+)\s*(?:-|đến)\s*(\d+)\s*(?:năm|year)', exp_lower): return f"{m.group(1)}-{m.group(2)} năm"
    if m := re.search(r'^(\d+)\s*(?:năm|year)', exp_lower): return f"{m.group(1)} năm"
    if m := re.search(r'(?:trên|>)\s*(\d+)\s*(?:năm|year)', exp_lower): return f"> {m.group(1)} năm"
    return exp_str # Trả về gốc nếu không khớp
print("Cell 11: Các hàm helper xử lý dữ liệu (`parse_salary_string_for_list_data`, `standardize_experience_for_list_data`) đã được định nghĩa.")

In [ ]:
# Cell 12: Data Processing - Apply Helper Functions (cho dữ liệu từ List Page) - HOÀN CHỈNH

if not df_jobs.empty:
    print("\nCell 12: Đang xử lý các cột gốc từ trang list (salary, post_date, experience)...")

    # --- KHỞI TẠO CÁC CỘT SẼ ĐƯỢC TẠO RA TỪ XỬ LÝ (ĐỂ ĐẢM BẢO LUÔN TỒN TẠI) ---
    # Cho Salary từ list
    df_jobs['salary_min_mil_list'] = np.nan
    df_jobs['salary_max_mil_list'] = np.nan
    df_jobs['salary_currency_list'] = 'triệu VNĐ' # Mặc định
    df_jobs['salary_type_list'] = 'Không xác định' # Mặc định
    # Cho Post Date từ list
    df_jobs['calculated_post_date_list'] = pd.NaT 
    # Cho Experience từ list
    df_jobs['experience_standardized_list'] = 'Không xác định' 

    # --- 1. Xử lý cột 'salary' (cột gốc tên là 'salary' trong df_jobs) ---
    if 'salary' in df_jobs.columns:
        salary_cols_parsed_names = ['salary_min_mil_list', 'salary_max_mil_list', 'salary_currency_list', 'salary_type_list']
        temp_salary_parsed_df = df_jobs['salary'].apply(
            lambda x: pd.Series(parse_salary_string_for_list_data(x), index=salary_cols_parsed_names)
        )
        for col_name in salary_cols_parsed_names:
            df_jobs[col_name] = temp_salary_parsed_df[col_name]
        print(f"  Đã xử lý cột 'salary' (gốc từ list). Các cột được cập nhật: {', '.join(salary_cols_parsed_names)}")
    else:
        print("  Cảnh báo Cell 12: Cột 'salary' (gốc từ list) không tồn tại. Các cột salary_..._list sẽ giữ giá trị NaN/mặc định.")

    # --- 2. Xử lý 'post_date' (cột gốc tên là 'post_date' trong df_jobs) ---
    if 'post_date' in df_jobs.columns and 'scrape_date' in df_jobs.columns:
        for index, row in df_jobs.iterrows():
            date_str, scrape_str = row['post_date'], row['scrape_date'] 
            calc_date = pd.NaT 
            if pd.isna(date_str) or pd.isna(scrape_str): continue
            try: scrape_dt = datetime.strptime(str(scrape_str), '%Y-%m-%d')
            except ValueError: continue
            date_lower = str(date_str).lower()
            if "hôm qua" in date_lower: calc_date = scrape_dt - timedelta(days=1)
            elif "hôm nay" in date_lower: calc_date = scrape_dt
            elif "ngày trước" in date_lower and (m := re.search(r'(\d+)', date_lower)): calc_date = scrape_dt - timedelta(days=int(m.group(1)))
            elif "tuần trước" in date_lower and (m := re.search(r'(\d+)', date_lower)): calc_date = scrape_dt - timedelta(weeks=int(m.group(1)))
            elif "tháng trước" in date_lower and (m := re.search(r'(\d+)', date_lower)): calc_date = scrape_dt - timedelta(days=int(int(m.group(1)) * 30.4375))
            else: 
                try: calc_date = pd.to_datetime(date_str, dayfirst=True, errors='coerce')
                except: pass 
                if pd.isna(calc_date):
                    try: calc_date = pd.to_datetime(date_str, errors='coerce')
                    except: pass                    
            if pd.notna(calc_date): 
                df_jobs.loc[index, 'calculated_post_date_list'] = calc_date.normalize() if isinstance(calc_date, pd.Timestamp) else calc_date.replace(hour=0,minute=0,second=0,microsecond=0)
        print("  Đã xử lý 'post_date' (gốc từ list). Cột được cập nhật: 'calculated_post_date_list'")
    else:
        print("  Cảnh báo Cell 12: Thiếu cột 'post_date' hoặc 'scrape_date'. 'calculated_post_date_list' sẽ là NaT.")

    # --- 3. Xử lý 'experience' (cột gốc tên là 'experience' trong df_jobs) ---
    if 'experience' in df_jobs.columns:
        df_jobs['experience_standardized_list'] = df_jobs['experience'].apply(standardize_experience_for_list_data)
        print("  Đã xử lý 'experience' (gốc từ list). Cột được cập nhật: 'experience_standardized_list'")
    else:
        print("  Cảnh báo Cell 12: Cột 'experience' (gốc từ list) không tồn tại. 'experience_standardized_list' sẽ là 'Không xác định'.")
    
    # --- 4. ĐỔI TÊN CÁC CỘT GỐC TỪ LIST PAGE (nếu tồn tại) ĐỂ CÓ HẬU TỐ _raw_list ---
    cols_to_rename_to_raw = {
        'company_name': 'company_name_raw_list', 
        'salary': 'salary_raw_list',         
        'location_raw': 'location_raw_list', 
        'city_main': 'city_main_raw_list',       
        'district': 'district_raw_list',         
        'experience': 'experience_raw_list', 
        'post_date': 'post_date_raw_list'    
    }
    
    actual_renamed_cols = []
    for old_name, new_name in cols_to_rename_to_raw.items():
        if old_name in df_jobs.columns:
            df_jobs.rename(columns={old_name: new_name}, inplace=True)
            actual_renamed_cols.append(new_name)
        else:
            # Nếu cột gốc không tồn tại, đảm bảo cột _raw_list vẫn được tạo với giá trị None
            # để schema DB không bị lỗi khi thiếu cột
            if new_name not in df_jobs.columns: # Chỉ tạo nếu chưa có
                 df_jobs[new_name] = None
                 print(f"  Cảnh báo Cell 12: Cột gốc '{old_name}' không tìm thấy. Tạo cột '{new_name}' với giá trị None.")
            
    if actual_renamed_cols:
        print(f"  Đã đổi tên các cột gốc từ list page thành dạng _raw_list: {', '.join(actual_renamed_cols)}")
        
    print("  Hoàn tất xử lý và đổi tên cột cho dữ liệu từ trang list.")
else:
    print("Cell 12: DataFrame 'df_jobs' rỗng. Bỏ qua xử lý.")

In [ ]:
# Cell 13: Data Processing - Apply Transformations (cho dữ liệu từ Detail Page)

if not df_jobs.empty:
    print("\nCell 13: Đang xử lý bổ sung cho các cột từ trang chi tiết...")

    # 1. Chuẩn hóa 'quantity_needed'
    if 'quantity_needed' in df_jobs.columns:
        df_jobs['quantity_needed_parsed'] = df_jobs['quantity_needed'].apply(
            lambda x: int(re.search(r'\d+', str(x)).group(0)) if pd.notna(x) and isinstance(x, str) and re.search(r'\d+', str(x)) else (int(x) if pd.notna(x) and isinstance(x, (int, float)) else None)
        )
        print("  Đã xử lý 'quantity_needed'. Cột mới: 'quantity_needed_parsed'")
    else:
        df_jobs['quantity_needed_parsed'] = None # Đảm bảo cột tồn tại
        print("  Cảnh báo: Cột 'quantity_needed' không tồn tại. 'quantity_needed_parsed' được tạo với giá trị None.")

    # 2. Chuẩn hóa 'application_deadline_date'
    if 'application_deadline_date' in df_jobs.columns:
        df_jobs['application_deadline_dt'] = pd.to_datetime(df_jobs['application_deadline_date'], format='%d/%m/%Y', errors='coerce')
        print("  Đã xử lý 'application_deadline_date'. Cột mới: 'application_deadline_dt' (datetime)")
    else:
        df_jobs['application_deadline_dt'] = pd.NaT # Đảm bảo cột tồn tại
        print("  Cảnh báo: Cột 'application_deadline_date' không tồn tại. 'application_deadline_dt' được tạo với giá trị NaT.")

    # --- Hàm join_list_elements (đã định nghĩa ở lần sửa trước, đảm bảo nó có ở đây hoặc cell trước) ---
    def join_list_elements(data_list):
        if isinstance(data_list, list):
            return ', '.join(str(item) for item in data_list) if data_list else None
        elif isinstance(data_list, np.ndarray) and data_list.size == 0: return None
        elif pd.isna(data_list): return None
        return str(data_list) if data_list else None

    # 3. Join list các skills/categories thành string
    list_cols_to_join_str = ['related_job_categories', 'required_skills_tags', 'preferred_skills_tags']
    for col_name in list_cols_to_join_str:
        if col_name in df_jobs.columns:
            df_jobs[f'{col_name}_str'] = df_jobs[col_name].apply(join_list_elements)
            print(f"  Đã chuyển cột list '{col_name}' thành string '{col_name}_str'")
        else:
            df_jobs[f'{col_name}_str'] = None # Đảm bảo cột tồn tại
            print(f"  Cảnh báo: Cột list '{col_name}' không tồn tại. '{col_name}_str' được tạo với giá trị None.")
    
    # --- 4. Trích xuất kỹ năng từ text JD/Requirements ---
    MASTER_SKILL_LIST = [
        "python", "sql", "java", "scala", "r", "c++", "c#", "javascript", "typescript", "php", "swift", "kotlin", "go", "ruby", "rust",
        "pandas", "numpy", "scipy", "scikit-learn", "sklearn", "tensorflow", "keras", "pytorch", "opencv", "nltk", "spacy", "gensim",
        "spark", "pyspark", "hadoop", "mapreduce", "hive", "pig", "kafka", "flink", "storm",
        "airflow", "luigi", "azkaban", "nifi",
        "docker", "kubernetes", "k8s", "openshift", "ansible", "terraform", "jenkins", "git", "cicd", "ci/cd",
        "aws", "azure", "gcp", "cloud", "s3", "ec2", "lambda", "rds", "dynamodb", "redshift", "glue", "emr", "sagemaker", 
        "azure data factory", "azure databricks", "azure synapse", "google bigquery", "google dataflow", "google dataproc",
        "excel", "vba", "power bi", "powerbi", "tableau", "qlik", "qlikview", "qliksense", "google data studio", "superset", "metabase", "looker",
        "ssis", "ssas", "ssrs", "informatica", "talend", "datastage",
        "etl", "elt", "data pipeline", "data modeling", "data modelling", "data warehouse", "dwh", "data lake", "data mart", "olap",
        "data analysis", "data analyst", "business intelligence", "bi", "business analyst", "ba",
        "data visualization", "data mining", "machine learning", "ml", "deep learning", "dl", "artificial intelligence", "ai", "nlp", "natural language processing",
        "statistics", "statistical modeling", "A/B testing", "hypothesis testing",
        "api", "restful api", "graphql", "microservices",
        "linux", "unix", "bash", "shell scripting",
        "nosql", "mongodb", "cassandra", "redis", "elasticsearch", "neo4j",
        "agile", "scrum", "kanban",
        "english", "tiếng anh", "giao tiếp", "communication", "presentation", "trình bày", 
        "problem solving", "critical thinking", "analytical skills", "phân tích", "tư duy logic", "teamwork", "làm việc nhóm"
    ] 
    
    df_jobs['skills_from_jd_text_list'] = None 
    df_jobs['skills_from_jd_text_str'] = None  

    if 'job_description_text' in df_jobs.columns or 'job_requirements_text' in df_jobs.columns:
        def extract_master_skills(row, skill_list_master):
            text_jd = str(row.get('job_description_text','')) if pd.notna(row.get('job_description_text')) else ''
            text_req = str(row.get('job_requirements_text','')) if pd.notna(row.get('job_requirements_text')) else ''
            
            # Không cần kết hợp tag skill ở đây nữa vì chúng ta đã có cột riêng cho chúng
            text_combined = (text_jd + " " + text_req).lower()
            
            if not text_combined.strip(): return None
            
            found_skills = []
            for skill_keyword in skill_list_master:
                # Regex để khớp từ/cụm từ hoàn chỉnh, không phân biệt hoa thường
                pattern = r'\b' + re.escape(skill_keyword.lower()) + r'\b'
                if re.search(pattern, text_combined):
                    found_skills.append(skill_keyword) 
            
            return list(set(found_skills)) if found_skills else None
        
        df_jobs['skills_from_jd_text_list'] = df_jobs.apply(extract_master_skills, args=(MASTER_SKILL_LIST,), axis=1)
        
        if 'skills_from_jd_text_list' in df_jobs.columns: # Kiểm tra lại trước khi apply
             df_jobs['skills_from_jd_text_str'] = df_jobs['skills_from_jd_text_list'].apply(join_list_elements)
        print("  Đã trích xuất skills từ text JD. Cột mới: 'skills_from_jd_text_list', 'skills_from_jd_text_str'")
    else:
        print("  Cảnh báo: Không có 'job_description_text' hoặc 'job_requirements_text' để trích xuất skill từ text. 'skills_from_jd_text_str' sẽ là None.")

    print("  Hoàn tất xử lý dữ liệu từ trang chi tiết.")
else:
    print("Cell 13: DataFrame 'df_jobs' rỗng. Bỏ qua xử lý.")

In [ ]:
# Cell 14: Database Operations - Schema, Connection, Insertion

load_dotenv() 

DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_TABLE_NAME = 'topcv_jobs_detailed' 

CREATE_TABLE_FINAL_SCHEMA_SQL = f"""
CREATE TABLE IF NOT EXISTS {DB_TABLE_NAME} (
    job_id SERIAL PRIMARY KEY,
    job_title TEXT,
    job_url TEXT UNIQUE,
    
    company_name_raw_list TEXT,
    salary_raw_list TEXT,
    salary_min_mil_list NUMERIC(10, 2),
    salary_max_mil_list NUMERIC(10, 2),
    salary_currency_list TEXT,
    salary_type_list TEXT,
    location_raw_list TEXT,
    city_main_raw_list TEXT,
    district_raw_list TEXT,
    experience_raw_list TEXT,
    experience_standardized_list TEXT,
    post_date_raw_list TEXT,
    calculated_post_date_list DATE,
    scrape_date DATE,

    company_name_detail TEXT, 
    company_scale TEXT,
    company_field TEXT,
    company_full_address TEXT,

    job_level TEXT,
    education_level TEXT,
    quantity_needed_parsed INT,
    employment_type TEXT,
    gender_requirement TEXT,

    related_job_categories_str TEXT,
    required_skills_tags_str TEXT,
    preferred_skills_tags_str TEXT,
    skills_from_jd_text_str TEXT,     -- THÊM CỘT NÀY

    job_description_text TEXT,
    job_requirements_text TEXT,
    job_benefits_text TEXT,
    working_time_text TEXT,
    
    application_deadline_dt DATE,
    
    inserted_at TIMESTAMP WITHOUT TIME ZONE DEFAULT CURRENT_TIMESTAMP 
);
"""

if not df_jobs.empty:
    print(f"\nCell 14: Đang chuẩn bị lưu dữ liệu vào DB bảng '{DB_TABLE_NAME}'...")
    if not all([DB_NAME, DB_USER, DB_PASSWORD]):
        print("  Lỗi: Thiếu thông tin kết nối database. Kiểm tra file .env.")
    else:
        conn, cur = None, None
        try:
            conn = psycopg2.connect(dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD, host=DB_HOST, port=DB_PORT)
            conn.autocommit = False 
            cur = conn.cursor()
            print("  Kết nối DB thành công!")
            
            cur.execute(CREATE_TABLE_FINAL_SCHEMA_SQL) # Chạy schema mới
            print(f"  Bảng '{DB_TABLE_NAME}' đã sẵn sàng.")

            df_for_db_insert = df_jobs.copy()
            
            db_target_columns = [
                'job_title', 'job_url', 
                'company_name_raw_list', 'salary_raw_list', 
                'salary_min_mil_list', 'salary_max_mil_list', 'salary_currency_list', 'salary_type_list',
                'location_raw_list', 'city_main_raw_list', 'district_raw_list',
                'experience_raw_list', 'experience_standardized_list',
                'post_date_raw_list', 'calculated_post_date_list', 'scrape_date',
                'company_name_detail', 'company_scale', 'company_field', 'company_full_address',
                'job_level', 'education_level', 'quantity_needed_parsed', 'employment_type', 'gender_requirement',
                'related_job_categories_str', 'required_skills_tags_str', 'preferred_skills_tags_str',
                'skills_from_jd_text_str', # THÊM CỘT NÀY VÀO LIST
                'job_description_text', 'job_requirements_text', 'job_benefits_text', 'working_time_text',
                'application_deadline_dt'
            ]
            
            missing_cols_in_df = [col for col in db_target_columns if col not in df_for_db_insert.columns]
            if missing_cols_in_df:
                print(f"  !!! LỖI: Các cột sau không tìm thấy trong DataFrame để chèn vào DB: {missing_cols_in_df}")
                print(f"      Các cột hiện có trong DataFrame: {sorted(list(df_for_db_insert.columns))}")
                # Tạo các cột thiếu với giá trị None để tránh lỗi insert, nhưng cần kiểm tra lại logic ở các cell trước
                for col_to_add in missing_cols_in_df:
                    print(f"      Tạo cột '{col_to_add}' với giá trị None do bị thiếu.")
                    df_for_db_insert[col_to_add] = None
                # raise KeyError(f"Missing columns for DB insert: {missing_cols_in_df}") # Hoặc raise lỗi để dừng lại

            date_cols_for_db = ['calculated_post_date_list', 'scrape_date', 'application_deadline_dt']
            for col_dt_db in date_cols_for_db:
                if col_dt_db in df_for_db_insert.columns:
                    df_for_db_insert[col_dt_db] = pd.to_datetime(df_for_db_insert[col_dt_db], errors='coerce').dt.strftime('%Y-%m-%d')
                    df_for_db_insert[col_dt_db] = df_for_db_insert[col_dt_db].replace({pd.NaT: None, 'NaT': None, 'nan':None})

            list_of_tuples_for_db = [
                tuple(row[col_name] if pd.notna(row[col_name]) else None for col_name in db_target_columns)
                for _, row in df_for_db_insert.iterrows()
            ]

            if not list_of_tuples_for_db:
                print("  Không có dữ liệu đã xử lý để chèn vào database.")
            else:
                print(f"  Đã chuẩn bị {len(list_of_tuples_for_db)} dòng dữ liệu để chèn/cập nhật.")
                insert_stmt = sql.SQL("INSERT INTO {} ({}) VALUES %s ON CONFLICT (job_url) DO NOTHING;").format(
                    sql.Identifier(DB_TABLE_NAME),
                    sql.SQL(', ').join(map(sql.Identifier, db_target_columns))
                )
                
                execute_values(cur, insert_stmt.as_string(conn), list_of_tuples_for_db, page_size=100)
                print(f"  Đã chèn/bỏ qua (nếu trùng job_url): {cur.rowcount} dòng.")
                conn.commit()
                print("  Dữ liệu đã được commit vào database.")

        except psycopg2.Error as e_db: print(f"  Lỗi PostgreSQL: {e_db}"); conn.rollback() if conn else None
        except KeyError as e_key: print(f"  Lỗi KeyError khi chuẩn bị chèn vào DB: {e_key}")
        except Exception as e_gen: print(f"  Lỗi không xác định khi thao tác DB: {e_gen}"); conn.rollback() if conn else None
        finally:
            if cur: cur.close()
            if conn: conn.close(); print("  Đã đóng kết nối database.")
else:
    print("Cell 14: DataFrame 'df_jobs' rỗng. Không có gì để thao tác với database.")

print("\n--- HOÀN TẤT TOÀN BỘ QUÁ TRÌNH NOTEBOOK ---")